#### Install Requirements

In [ ]:
! pip install -r ../requirements.txt

#### Install custom package

In [ ]:
! pip uninstall cnn-transfer -y # uninstall pre existing package
! pip install -e ..

#### Imports

In [3]:
import os
import json
import torch
import torch.nn as nn
import pandas as pd

from cnn_transfer.dataset import *
from cnn_transfer.models import *
from cnn_transfer.evaluate import *
from cnn_transfer.corruption import *
from cnn_transfer.feature_probe import *
from cnn_transfer.utils import set_seed

#### Model

* Models implemented for this assignment : resnet50, densenet121, efficientnet_b0
* Change the MODEL_NAME to "resnet50" / " densenet121" / "efficientnet_b0" as desired
* Change the EVAL_SPLIT to  "val" / "test" # change to "test" if test set is used

In [4]:
MODEL_NAME = "resnet50"   # change this as desired
EVAL_SPLIT = "val" # change to "test" if test set is used

#### Configuration

In [5]:
config = {
    "model_name": MODEL_NAME,
    "num_classes": 30,
    "batch_size": 64,
    "img_size": 224
}

#### Setup Paths

In [6]:
BASE_DIR = ".."

DATA_DIR = os.path.join(BASE_DIR, "dataset", "train_data")

LOG_DIR = os.path.join(BASE_DIR, "logs", MODEL_NAME)

CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints", MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_seed(42)

In [7]:
save_folders = [
    "results",
    "plots",
]

models = [
    "resnet50",
    "densenet121",
    "efficientnet_b0"
]

for i in save_folders:
  for j in models:
    folder_path = os.path.join(BASE_DIR, i)
    path = os.path.join(folder_path, j)
    os.makedirs(path, exist_ok=True)

#### Load Dataset

In [ ]:
dataset, train_t, val_t = load_dataset(
    DATA_DIR,
    config["img_size"]
)

train_dataset, val_dataset = create_train_val_split(dataset)

train_dataset.dataset.transform = train_t
val_dataset.dataset.transform = val_t

train_loader, val_loader = create_dataloaders(
    train_dataset,
    val_dataset,
    config["batch_size"]
)

#### Test loader

In [ ]:
TEST_DIR = os.path.join(BASE_DIR, "dataset", "test_data")

if os.path.exists(TEST_DIR):

    from torchvision.datasets import ImageFolder
    from torch.utils.data import DataLoader

    test_dataset = ImageFolder(TEST_DIR, transform=val_t)

    test_loader = DataLoader(
        test_dataset,
        batch_size=config["batch_size"],
        shuffle=False
    )

else:
    test_loader = None

#### Select Loader

In [ ]:
if EVAL_SPLIT == "val":
    eval_loader = val_loader
elif EVAL_SPLIT == "test" and test_loader is not None:
    eval_loader = test_loader
else:
    raise ValueError("Test loader not available")

## Scenario 4.1 Evaluation

#### Load Model

In [ ]:
model = load_model(
    config["model_name"],
    config["num_classes"]
)

model_path = os.path.join(
    CHECKPOINT_DIR,
    "4.1_model_final.pt"
)

model.load_state_dict(torch.load(model_path, map_location=device))

model = model.to(device)

model.eval()

#### Load History

In [ ]:
history_file = os.path.join(LOG_DIR, "4.1_history.json")

with open(history_file) as f:
    history = json.load(f)

#### Accuracy Curves


In [ ]:
# Training and validation accuracy curves
from cnn_transfer.evaluate import plot_train_val_acc_curves_and_save
plot_train_val_acc_curves_and_save(history, config['model_name'], BASE_DIR)

#### Confusion Matrix

In [ ]:
from cnn_transfer.evaluate import plot_confusion_matrix_and_save
plot_confusion_matrix_and_save(model, eval_loader, device, BASE_DIR, config["model_name"], dataset)

#### Feature Embedding Visualization

In [ ]:
from cnn_transfer.evaluate import extract_features
features, labels = extract_features(model,eval_loader,device)

#### PCA, t-SNE

In [ ]:
# PCA
from cnn_transfer.evaluate import plot_pca
plot_pca(features, labels, BASE_DIR, config["model_name"])

In [ ]:
# t-SNE
from cnn_transfer.evaluate import plot_tsne
plot_tsne(features, labels, BASE_DIR, config["model_name"])

## Scenario 4.2 Fine-Tuning Evaluation

In [ ]:
if MODEL_NAME == "resnet50":


    percent_unfrozen = {
    "linear_probe": 0.26,
    "selective_20": 19.21,
    "last_block": 63.75,
    "full_ft": 100
    }

elif MODEL_NAME == "densenet121":

    layers = {
        "early": model.features.denseblock1,
        "middle": model.features.denseblock3,
        "final": model.features.denseblock4
    }

elif MODEL_NAME == "efficientnet_b0":

    layers = {
        "early": model.blocks[1],
        "middle": model.blocks[4],
        "final": model.blocks[-1]
    }

In [ ]:
histories = {}

files = {
    "linear_probe": "4.2_linear_probe_history.json",
    "selective_20": "4.2_selective_20_unfreeze_history.json",
    "last_block": "4.2_last_block_csf_unfreeze_history.json",
    "full_ft": "4.2_full_finetune_history.json"}

for k, f in files.items():
    with open(os.path.join(LOG_DIR, f)) as file:
        histories[k] = json.load(file)

In [ ]:
percent_unfrozen = percent_unfrozen

####  Training and Validation Accuracy vs % Parameters

In [ ]:
from cnn_transfer.evaluate import plot_train_val_acc_percentage_param
plot_train_val_acc_percentage_param(histories, percent_unfrozen,BASE_DIR,config["model_name"])

####  Convergence Stability (Training Loss vs Epoch)

In [ ]:
from cnn_transfer.evaluate import plot_train_loss_vs_epoch
plot_train_loss_vs_epoch(histories,BASE_DIR,config["model_name"])

####  Gradient Norm Statistics

In [ ]:
from cnn_transfer.evaluate import grad_norm_stats
grad_norm_stats(histories,BASE_DIR,config["model_name"])

#### Summary Table

In [ ]:
from cnn_transfer.evaluate import summary_table
summary_table(histories, percent_unfrozen,BASE_DIR,config["model_name"])

## Scenario 4.3 Few-Shot Evaluation

#### Load Few-Shot Logs

In [ ]:
fractions = [1.0, 0.2, 0.05]

fewshot_results = {}

for frac in fractions:

    file = os.path.join(LOG_DIR,f"4.3_few_shot_{frac}.json")

    with open(file) as f:
        fewshot_results[frac] = json.load(f)

##### Validation Accuracy

In [ ]:
acc_100 = fewshot_results[1.0]["val_acc"][-1]
acc_20  = fewshot_results[0.2]["val_acc"][-1]
acc_5   = fewshot_results[0.05]["val_acc"][-1]

print(f"Val_Acc_100:{acc_100:.4f}, Val_Acc_20:{acc_20:.4f}, Val_Acc_5:{acc_5:.4f}")

In [ ]:
from cnn_transfer.evaluate import plot_val_acc_across_epochs
plot_val_acc_across_epochs(fewshot_results,BASE_DIR,config["model_name"])

##### Relative Performance Drop

In [ ]:
delta = (acc_100 - acc_5) / acc_100
print(f"Relative performance drop:{delta:.4f}")

##### Training–Validation Gap

In [ ]:
train_acc = fewshot_results[frac]["train_acc"][-1]
val_acc   = fewshot_results[frac]["val_acc"][-1]

gap = train_acc - val_acc
print(f"Training–validation gap:{gap:.4f}")

In [ ]:
from cnn_transfer.evaluate import train_val_acc_gap_acroos_epochs
train_val_acc_gap_acroos_epochs(fewshot_results,BASE_DIR,config["model_name"])

##### Plot Data Efficiency(Accuracy vs Data Fraction)

In [ ]:
data_sizes = [100, 20, 5]
accs = [acc_100, acc_20, acc_5]

from cnn_transfer.evaluate import plot_acc_vs_data_frac
plot_acc_vs_data_frac(fewshot_results,data_sizes,accs,BASE_DIR,config["model_name"])

## Scenario 4.4 Corruption Robustness

#### Load Fully Fine-Tuned Model

In [ ]:
model = load_model(
    config["model_name"],
    config["num_classes"]
)

model_path = os.path.join(
    CHECKPOINT_DIR,
    "4.2_full_finetune_model.pt"
)

model.load_state_dict(torch.load(model_path, map_location=device))

model = model.to(device)
model.eval()

In [ ]:
from cnn_transfer.corruption import corruption_robustness_analysis
criterion = torch.nn.CrossEntropyLoss().to(device)
results, acc_clean = corruption_robustness_analysis(model, eval_loader, criterion, device)

In [ ]:
import pandas as pd

df = pd.DataFrame(results).T

results_dir = os.path.join(BASE_DIR, "results")
model_dir = os.path.join(results_dir, config["model_name"])
df.to_csv(os.path.join(model_dir, "4.4_corruption_robustness_analysis_results.csv"), index=False)
df

## 4.5 Layer-Wise Feature Probing

In [ ]:
model = load_model(config["model_name"], config["num_classes"])
model = model.to(device)
model.eval()

In [ ]:
if MODEL_NAME == "resnet50":

    layers = {
        "early": model.layer1,
        "middle": model.layer3,
        "final": model.layer4
    }


elif MODEL_NAME == "densenet121":

    layers = {
        "early": model.features.denseblock1,
        "middle": model.features.denseblock3,
        "final": model.features.denseblock4
    }

elif MODEL_NAME == "efficientnet_b0":

    layers = {
        "early": model.blocks[1],
        "middle": model.blocks[4],
        "final": model.blocks[-1]
    }

In [ ]:
layers = layers
depth_acc = []
norm_stats = []
layer_features = {}

##### Extract Features + Train Probe

In [ ]:
for name, layer in layers.items():

    train_feats, train_labels = extract_layer_features(
        model,
        train_loader,
        device,
        layer
    )

    eval_feats, eval_labels = extract_layer_features(
        model,
        eval_loader,
        device,
        layer
    )

    # store features for PCA
    layer_features[name] = eval_feats

    acc = train_linear_probe(
        train_feats,
        train_labels,
        eval_feats,
        eval_labels
    )

    depth_acc.append(acc)

    mean_norm, std_norm = compute_feature_norms(eval_feats)

    norm_stats.append((mean_norm, std_norm))

##### Plot Accuracy vs Depth

In [ ]:
from cnn_transfer.feature_probe import plot_acc_vs_depth
plot_acc_vs_depth(depth_acc,BASE_DIR,config["model_name"])

##### Feature Norm Statistics

In [ ]:
from cnn_transfer.feature_probe import plot_feature_norms
plot_feature_norms(norm_stats,BASE_DIR,config["model_name"])

##### PCA Visualization

In [ ]:
val_feats_early = layer_features["early"]
val_feats_mid   = layer_features["middle"]
val_feats_final = layer_features["final"]

def plot_pca(features, labels, BASE_DIR, model_name, layer_name):

    from sklearn.decomposition import PCA
    import matplotlib.pyplot as plt
    import os

    pca = PCA(n_components=2)

    reduced = pca.fit_transform(features)

    plt.figure(figsize=(8,6))

    scatter = plt.scatter(
        reduced[:,0],
        reduced[:,1],
        c=labels,
        cmap="tab20",
        s=10
    )

    plt.title(f"{model_name} - {layer_name} Layer PCA")

    plot_dir = os.path.join(BASE_DIR, "plots")
    model_dir = os.path.join(plot_dir, model_name)

    os.makedirs(model_dir, exist_ok=True)

    plt.savefig(
        os.path.join(
            model_dir,
            f"{model_name}_4.5_pca_{layer_name}.png"
        )
    )

    plt.show()

In [ ]:
subset_idx = sample_fixed_subset(eval_labels, samples_per_class=30)

subset_feats_early = val_feats_early[subset_idx]
subset_feats_mid   = val_feats_mid[subset_idx]
subset_feats_final = val_feats_final[subset_idx]

subset_labels = eval_labels[subset_idx]


plot_pca(
    subset_feats_early,
    subset_labels,
    BASE_DIR,
    config["model_name"],
    "early"
)

plot_pca(
    subset_feats_mid,
    subset_labels,
    BASE_DIR,
    config["model_name"],
    "middle"
)

plot_pca(
    subset_feats_final,
    subset_labels,
    BASE_DIR,
    config["model_name"],
    "final"
)